# Similarity Analysis — Real vs Synthetic Data

Notebook focused on similarity / memorization metrics. Each section contains a brief explanation of the metric, the computation and a visualization. At the end, all results are saved to a CSV.

## Configuration

In [ ]:
# Choose the dataset
# Options: 'adult' | 'bank_marketing' | 'california_housing' | 'cardiotocography' | 'compas' | 'loan_prediction'
DATASET = 'bank_marketing'

# LLM frameworks to evaluate (those present in sub-folders of synthetic-data/)
# LLM_FRAMEWORKS = ['great', 'tabula', 'taptap']
LLM_FRAMEWORKS = []

# LLMs to evaluate within each framework
# LLMS = ['distilgpt2', 'taptap_distill', 'qwen3_03B_distil']
LLMS = []

# GANs to evaluate (empty to focus only on the LLMs)
# GANS = []
GANS = ['ctgan', 'tvae', 'tabfairgan', 'ctabgan', 'ctabgan_plus']

# Focal threshold for summary tables (the full curve is computed in section 5)
FOCAL_THRESHOLD = 0.8

# Dense list of thresholds for the curve
# CURVE_THRESHOLDS = [1.0, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5]
CURVE_THRESHOLDS = [1.0, 0.9, 0.8]


In [ ]:
import sys, warnings, os
warnings.filterwarnings('ignore')

sys.path.insert(0, '/Users/joaomonteiro/Desktop/Tese/datasets')
import synth_eval as se

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Per-dataset configs (file suffix + seeds + columns to drop at load time)
# cols_to_drop: columns that act as identifiers (dates, IDs, free text)
# and that distort row-level comparisons. Applied to real_train, real_test and all
# the synthetic data of the dataset, without needing to regenerate the synthetic data.
CONFIGS = {
    'adult':              {'dataset_name': 'adult',              'seeds': [42, 43, 44],         'cols_to_drop': []},
    'bank-marketing':     {'dataset_name': 'bank_marketing',     'seeds': [42, 43, 44],         'cols_to_drop': []},
    'california-housing': {'dataset_name': 'california_housing', 'seeds': [42, 43, 44],         'cols_to_drop': []},
    'cardiotocography':   {'dataset_name': 'cardiotocography',   'seeds': [42, 43, 44],         'cols_to_drop': []},
    'compas':             {'dataset_name': 'compas',             'seeds': [42, 43, 44],         'cols_to_drop': ['compas_screening_date', 'dob', 'in_custody', 'out_custody', 'c_charge_desc']},
    'loan-predication':   {'dataset_name': 'loan_predication',   'seeds': [42, 43, 44, 45, 46], 'cols_to_drop': []},
}

cfg          = CONFIGS[DATASET]
dataset_name = cfg['dataset_name']
seeds        = cfg['seeds']
cols_to_drop = cfg.get('cols_to_drop', [])
methods      = ['real'] + LLM_FRAMEWORKS + GANS

BASE_DIR      = f'/Users/joaomonteiro/Desktop/Tese/datasets/{DATASET}'
DATA_DIR      = f'{BASE_DIR}/data'
SYNTHETIC_DIR = f'{BASE_DIR}/synthetic-data'
RESULTS_DIR   = f'{BASE_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Dataset       : {DATASET}')
print(f'Seeds         : {seeds}')
print(f'Cols to drop  : {cols_to_drop}')
print(f'Frameworks    : {LLM_FRAMEWORKS}')
print(f'LLMs          : {LLMS}')
print(f'GANs          : {GANS}')


## Load data

In [ ]:
def _load(path):
    df = pd.read_csv(path)
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop, errors='ignore')
    return df

train_datasets = {'real': {}}
test_datasets  = {}

for seed in seeds:
    train_datasets['real'][seed] = _load(f'{DATA_DIR}/{dataset_name}_seed{seed}_train.csv')
    test_datasets[seed]          = _load(f'{DATA_DIR}/{dataset_name}_seed{seed}_test.csv')

for fw in LLM_FRAMEWORKS:
    train_datasets[fw] = {}
    for llm in LLMS:
        train_datasets[fw][llm] = {}
        for seed in seeds:
            path = f'{SYNTHETIC_DIR}/{fw}/{llm}/{dataset_name}_{fw}_{llm}_seed{seed}.csv'
            train_datasets[fw][llm][seed] = _load(path)

for gan in GANS:
    train_datasets[gan] = {}
    for seed in seeds:
        path = f'{SYNTHETIC_DIR}/{gan}/{dataset_name}_{gan}_seed{seed}.csv'
        train_datasets[gan][seed] = _load(path)

if dataset_name == 'bank_marketing' and 'tabfairgan' in GANS:
    for seed in seeds:
        train_datasets['tabfairgan'][seed].drop('age_group', axis=1, inplace=True, errors='ignore')

print(f'Train shape: {train_datasets["real"][seeds[0]].shape}')
print(f'Test shape : {test_datasets[seeds[0]].shape}')
print(f'Columns    : {list(train_datasets["real"][seeds[0]].columns)}')

### Cleaning of invalid categorical values
Before measuring similarity, we map synthetic categorical values that do not exist in the real data (e.g.: 'admin. primary' → 'admin.') to avoid artificially inflating the distances.

In [ ]:
fix_report = se.fix_invalid_values_in_train_datasets(train_datasets, methods, seeds)
print(f'Total cells corrected: {fix_report["n_corrected"].sum()} | unfixed: {fix_report["n_unfixed"].sum()}')

## 1. Discriminator (Real vs Synthetic)
Trains a binary classifier to separate real rows from synthetic rows. Returns the AUC: **0.5 → indistinguishable**, **1.0 → perfectly separable**.

In parallel we compute the i.i.d. baseline — AUC of the same classifier separating real_train from real_test. Since they come from the same distribution, this baseline should be close to 0.5. Comparing the synthetic AUC with this baseline is more informative than looking at the synthetic AUC in isolation: the gap `disc_auc(syn) − disc_auc(real_test)` measures how far the synthetic data departs from a fresh real sample.

In [ ]:
print('Running discriminator evaluation...')
disc_df = se.run_discriminator_evaluation(
    train_datasets, methods, seeds, test_datasets=test_datasets,
)
display(disc_df.round(4))


In [ ]:
# Bar plot — mean AUC per (method, llm) with min/max bars across seeds
# Horizontal reference line = i.i.d. baseline (real_test vs real_train)
syn_df = disc_df[disc_df['method'] != 'real_test']
baseline_df = disc_df[disc_df['method'] == 'real_test']

agg = syn_df.groupby(['method', 'llm'])['disc_auc'].agg(['mean', 'min', 'max']).reset_index()
agg['label'] = agg.apply(lambda r: r['method'] if r['llm'] == '-' else f"{r['method']}/{r['llm']}", axis=1)

fig, ax = plt.subplots(figsize=(max(6, 0.5*len(agg)), 4))
x = np.arange(len(agg))
err_lo = agg['mean'] - agg['min']
err_hi = agg['max'] - agg['mean']
ax.bar(x, agg['mean'], yerr=[err_lo, err_hi], capsize=3, color='C0', label='syn vs real_train')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='ideal (0.5)')

if not baseline_df.empty:
    b_mean = baseline_df['disc_auc'].mean()
    b_min  = baseline_df['disc_auc'].min()
    b_max  = baseline_df['disc_auc'].max()
    ax.axhline(b_mean, color='C1', linestyle='--', linewidth=1.4,
               label=f'real_test vs real_train (i.i.d. baseline)  μ={b_mean:.3f}')
    ax.fill_between([-0.5, len(agg)-0.5], b_min, b_max, color='C1', alpha=0.12)
    ax.set_xlim(-0.5, len(agg)-0.5)

ax.set_xticks(x); ax.set_xticklabels(agg['label'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('disc_auc')
ax.set_ylim(0.4, 1.05)
ax.set_title('Discriminator AUC (mean across seeds)')
ax.legend(fontsize=8); fig.tight_layout(); plt.show()


## 2. Correlation Difference
Measures the mean absolute error (MAE) between the real and synthetic correlation matrices (numeric columns only, upper triangle). **Lower is better**: it indicates that the generator preserves the linear relationships between pairs of variables.

In parallel we compute the i.i.d. baseline — MAE between the correlation matrices of real_train and real_test. Since they come from the same distribution, the difference is just sampling noise; this baseline defines the "irreducible" MAE. Synthetic data with MAE close to the baseline faithfully preserves the correlation structure; MAE well above the baseline indicates distortion of the relationships between variables.

In [ ]:
print('Running correlation evaluation...')
corr_df = se.run_correlation_evaluation(
    train_datasets, methods, seeds, test_datasets=test_datasets,
)
display(corr_df.round(4))


In [ ]:
# Bar plot — mean corr_mae per (method, llm) with min/max bars across seeds
# Horizontal reference line = i.i.d. baseline (real_test vs real_train)
syn_df = corr_df[corr_df['method'] != 'real_test']
baseline_df = corr_df[corr_df['method'] == 'real_test']

agg = syn_df.groupby(['method', 'llm'])['corr_mae'].agg(['mean', 'min', 'max']).reset_index()
agg['label'] = agg.apply(lambda r: r['method'] if r['llm'] == '-' else f"{r['method']}/{r['llm']}", axis=1)

fig, ax = plt.subplots(figsize=(max(6, 0.5*len(agg)), 4))
x = np.arange(len(agg))
err_lo = agg['mean'] - agg['min']
err_hi = agg['max'] - agg['mean']
ax.bar(x, agg['mean'], yerr=[err_lo, err_hi], capsize=3, color='C2', label='syn vs real_train')

if not baseline_df.empty:
    b_mean = baseline_df['corr_mae'].mean()
    b_min  = baseline_df['corr_mae'].min()
    b_max  = baseline_df['corr_mae'].max()
    ax.axhline(b_mean, color='C1', linestyle='--', linewidth=1.4,
               label=f'real_test vs real_train (i.i.d. baseline)  μ={b_mean:.4f}')
    ax.fill_between([-0.5, len(agg)-0.5], b_min, b_max, color='C1', alpha=0.12)
    ax.set_xlim(-0.5, len(agg)-0.5)

ax.set_xticks(x); ax.set_xticklabels(agg['label'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('corr_mae (↓ better)')
ax.set_title('Mean correlation difference (real vs synthetic)')
ax.legend(fontsize=8); fig.tight_layout(); plt.show()


## 3. Identical Rows (Exact Matches)
Percentage of real rows that have an exact copy in the synthetic set. **Higher → more verbatim copies**, a direct sign of memorization. Typically close to 0% for GANs and rare even in LLMs over high-cardinality datasets; when it appears, it is the strongest privacy signal.

In [ ]:
print('Running exact-row-match evaluation...')
equal_df = se.run_percent_equal_rows_evaluation(train_datasets, methods, seeds)
display(equal_df.round(4))


In [ ]:
agg = equal_df.groupby(['method', 'llm'])['pct'].agg(['mean', 'min', 'max']).reset_index()
agg['label'] = agg.apply(lambda r: r['method'] if r['llm'] == '-' else f"{r['method']}/{r['llm']}", axis=1)

fig, ax = plt.subplots(figsize=(max(6, 0.5*len(agg)), 4))
x = np.arange(len(agg))
err_lo = agg['mean'] - agg['min']
err_hi = agg['max'] - agg['mean']
ax.bar(x, agg['mean'], yerr=[err_lo, err_hi], capsize=3, color='C3')
ax.set_xticks(x); ax.set_xticklabels(agg['label'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('% real rows with exact copy')
ax.set_title('Exact copies (real rows reproduced in synthetic)')
fig.tight_layout(); plt.show()


## 4. Distance to Closest Record (DCR)
For each synthetic row, computes the distance to the nearest real row (after min-max normalization of the columns by the real range). **Very low values** indicate synthetic rows placed on top of real rows — a privacy risk. We report mean, median, minimum and 5th percentile (left tail).

In [ ]:
print('Running DCR evaluation...')
dcr_df = se.run_dcr_evaluation(train_datasets, methods, seeds)
display(dcr_df.round(4))


In [ ]:
# Median (center of the distribution) and 5th percentile (left tail — risk)
agg = dcr_df.groupby(['method', 'llm'])[['dcr_median', 'dcr_5th']].mean().reset_index()
agg['label'] = agg.apply(lambda r: r['method'] if r['llm'] == '-' else f"{r['method']}/{r['llm']}", axis=1)

fig, axes = plt.subplots(1, 2, figsize=(max(10, 0.7*len(agg)), 4))
x = np.arange(len(agg))
axes[0].bar(x, agg['dcr_median'], color='C4'); axes[0].set_title('DCR median')
axes[1].bar(x, agg['dcr_5th'],    color='C1'); axes[1].set_title('DCR 5th percentile (tail)')
for ax in axes:
    ax.set_xticks(x); ax.set_xticklabels(agg['label'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('normalized distance (↓ closer to real)')
fig.tight_layout(); plt.show()


## 5. Similarity Curve vs Threshold
For each synthetic row it finds the **nearest** real row (after normalization) and computes the fraction of columns that match. We report the percentage of synthetic rows with `match-fraction ≥ t` for several thresholds *t*.

We compute the curve against **real_train** (solid), against **real_test** (dashed) and the i.i.d. baseline **real_test → real_train** (dotted). Reading:

- syn-vs-train **above** the dotted baseline → train-specific memorization.
- syn-vs-train **overlapping** the baseline → marginal coverage without memorization.
- syn-vs-train **below** the baseline → the generator underfits the train (typical of GANs).

In [ ]:
print('Running threshold curve syn-vs-train and syn-vs-test (may take a while)...')
curve_df = se.run_percentage_similar_rows_train_vs_test(
    train_datasets, test_datasets, methods, seeds,
    thresholds=CURVE_THRESHOLDS,
)
display(curve_df.round(2))


In [ ]:
fig = se.plot_similarity_threshold_curves(
    curve_df,
    title='% synthetic rows ≥ t — solid = vs train, dashed = vs test, dotted = i.i.d. baseline',
)
plt.show()


## 6. Memorization Premium
Difference in percentage points between `syn-vs-train` and the i.i.d. baseline (`real_test → real_train`) at each threshold. Measures how far **above a fresh real sample** the generator gets against the train. Premium ≈ 0 → no memorization; positive premium → memorization at that similarity tier.

In [ ]:
# Pivot: separate the baseline from the synthetic curve and compute the gap at each threshold
pct_cols = [c for c in curve_df.columns if c.startswith('pct_')]

baseline = (curve_df[curve_df['method'] == 'real_test']
            .set_index('seed')[pct_cols])

syn = (curve_df[(curve_df['method'] != 'real_test') & (curve_df['reference'] == 'train')]
       .copy())

for c in pct_cols:
    syn[f'premium_{c}'] = syn.apply(lambda r: r[c] - baseline.loc[r['seed'], c], axis=1)

premium_cols = [f'premium_{c}' for c in pct_cols]
premium_df = (syn.groupby(['method', 'llm'])[premium_cols]
                 .mean()
                 .reset_index()
                 .round(2))
display(premium_df)


In [ ]:
# Heatmap: premium at each threshold per (method, llm) — red = memorization
focal = f'premium_pct_{int(round(FOCAL_THRESHOLD * 100))}'

hm = premium_df.copy()
hm['label'] = hm.apply(lambda r: r['method'] if r['llm'] == '-' else f"{r['method']}/{r['llm']}", axis=1)
hm = hm.set_index('label')[premium_cols]
hm.columns = [int(c.split('_')[-1]) / 100 for c in hm.columns]
hm = hm[sorted(hm.columns, reverse=True)]

fig, axes = plt.subplots(1, 2, figsize=(14, max(3, 0.4*len(hm) + 1.5)),
                         gridspec_kw={'width_ratios': [3, 1]})

vmax = max(abs(hm.values.min()), abs(hm.values.max()))
im = axes[0].imshow(hm.values, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_xticks(np.arange(len(hm.columns))); axes[0].set_xticklabels([f'{t:.2f}' for t in hm.columns], fontsize=8)
axes[0].set_yticks(np.arange(len(hm.index)));  axes[0].set_yticklabels(hm.index, fontsize=8)
axes[0].set_xlabel('threshold t'); axes[0].set_title('Memorization premium (pp above the i.i.d. baseline)')
axes[0].grid(False)
fig.colorbar(im, ax=axes[0], fraction=0.025, pad=0.02)

# Right panel: bar plot of the focal premium
vals = hm[FOCAL_THRESHOLD]
axes[1].barh(np.arange(len(vals)), vals.values, color=['C3' if v > 0 else 'C0' for v in vals.values])
axes[1].set_yticks(np.arange(len(vals))); axes[1].set_yticklabels(vals.index, fontsize=8)
axes[1].invert_yaxis()
axes[1].axvline(0, color='gray', linewidth=0.8)
axes[1].set_title(f'Premium at t={FOCAL_THRESHOLD}')
axes[1].set_xlabel('pp above the baseline')
fig.tight_layout(); plt.show()


## 7. Per-Column Decomposition
For each synthetic row that reaches `match-fraction ≥ t`, identifies in which columns a match occurred. We compute:

- `hit_rate`: fraction of the matched rows where the column matched.
- `baseline_rate`: probability of that match by chance (independent sampling of the marginals).
- `lift = hit_rate / baseline_rate`. Lift close to 1 → low-entropy column ("matchable" structurally); very high lift in high-cardinality columns → row-level memorization.

To distinguish memorization from coverage, we compute the decomposition twice: using real_train as reference and using real_test. **lift_train ≫ lift_test** in a column signals train-specific memorization in that column.

In [ ]:
print(f'Running per-column decomposition at t={FOCAL_THRESHOLD}...')
decomp_df = se.run_match_column_contribution_train_vs_test(
    train_datasets, test_datasets, methods, seeds,
    threshold=FOCAL_THRESHOLD,
)

# Pivot: lift_train vs lift_test, gap descending
lift_pivot = (decomp_df
              .replace([np.inf, -np.inf], np.nan)
              .pivot_table(index=['method', 'llm', 'column', 'dtype'],
                           columns='reference', values='lift',
                           aggfunc='mean', dropna=False)
              .reset_index())
# Ensure both columns exist even in datasets where the test side collapses
for _ref in ('train', 'test'):
    if _ref not in lift_pivot.columns:
        lift_pivot[_ref] = np.nan
lift_pivot = lift_pivot.rename(columns={'train': 'lift_train', 'test': 'lift_test'})
lift_pivot = lift_pivot[['method', 'llm', 'column', 'dtype', 'lift_train', 'lift_test']]
lift_pivot['lift_gap'] = lift_pivot['lift_train'] - lift_pivot['lift_test']
lift_pivot = lift_pivot.sort_values('lift_gap', ascending=False)
print('Top 15 (method, column) by lift_gap:')
display(lift_pivot.head(15).round(2))


In [ ]:
# Safeguard: warn when n_matched_rows is too low for the per-column analysis
# to be statistically reliable (default: 50). If the warning fires, consider lowering
# the FOCAL_THRESHOLD and re-running this section.
MIN_MATCHED_ROWS = int(train_datasets["real"][42].shape[0]*0.5)

_sizes = se.check_match_sample_size(decomp_df, min_n=MIN_MATCHED_ROWS)
_bad   = _sizes[~_sizes['ok']]

if _bad.empty:
    print(f'OK — all combinations have n_matched_rows >= {MIN_MATCHED_ROWS}.')
else:
    print(f'⚠ Combinations with n_matched_rows < {MIN_MATCHED_ROWS} '
          f'(per-column hit_rate / lift are not reliable on these rows):')
    display(_bad)

    n_syn_seed0 = sum(
        len(train_datasets[m][LLMS[0]][seeds[0]]) if m in LLM_FRAMEWORKS
        else len(train_datasets[m][seeds[0]])
        for m in methods if m != 'real'
    ) // max(1, len([m for m in methods if m != 'real']))
    suggested_t = se.suggest_threshold_for_sample_size(
        curve_df, min_n=MIN_MATCHED_ROWS, n_syn_hint=n_syn_seed0,
    )
    if suggested_t is not None:
        print(f'\nSuggestion: re-run this section with FOCAL_THRESHOLD = {suggested_t:.2f} '
              f'(largest threshold with n_matched >= {MIN_MATCHED_ROWS} across all pairs).')
    else:
        print(f'\nNo threshold in curve_df reaches n_matched >= {MIN_MATCHED_ROWS} '
              f'for all pairs. The per-column analysis is not informative on this dataset; '
              f'report the global fidelity metrics (disc_auc, corr_mae, DCR) instead.')


In [ ]:
# Visual heatmap: lift against train (means across seeds)
fig = se.plot_match_column_contribution(
    decomp_df[decomp_df['reference'] == 'train'],
    value='lift', annotate=True,
    title=f'lift per column at t={FOCAL_THRESHOLD} (reference = real_train)',
)
plt.show()


### 7.1 hit_rate gap (robust summary across seeds)
The `lift` is sensitive to noise because the `baseline_rate` is estimated from 5000 random pairs and varies a lot across seeds for continuous columns (where the real probability of a coincidence is very small). Comparing `hit_rate` against train with `hit_rate` against test gives a more stable and directly interpretable signal:
- `hit_gap = hit_train − hit_test` in percentage points.
- `hit_ratio = hit_train / hit_test`.

Use the **lift** to pick suspicious columns (it rescales variables with very different cardinalities) and **hit_gap / hit_ratio** to report the magnitude of memorization in those columns.

In [ ]:
# Pivot hit_rate instead of lift and compute gap/ratio (mean across seeds)
hr_pivot = (decomp_df
            .pivot_table(index=['method', 'llm', 'column', 'dtype', 'seed'],
                         columns='reference', values='hit_rate',
                         aggfunc='mean', dropna=False)
            .reset_index())
for _ref in ('train', 'test'):
    if _ref not in hr_pivot.columns:
        hr_pivot[_ref] = np.nan
hr_pivot = hr_pivot.rename(columns={'train': 'hit_train', 'test': 'hit_test'})
hr_pivot['hit_gap']   = hr_pivot['hit_train'] - hr_pivot['hit_test']
hr_pivot['hit_ratio'] = hr_pivot['hit_train'] / hr_pivot['hit_test'].replace(0, np.nan)

# Means across seeds
hit_summary = (hr_pivot
               .groupby(['method', 'llm', 'column', 'dtype'])
               .agg(hit_train=('hit_train', 'mean'),
                    hit_test=('hit_test', 'mean'),
                    hit_gap=('hit_gap', 'mean'),
                    hit_ratio=('hit_ratio', 'mean'))
               .reset_index()
               .sort_values('hit_ratio', ascending=False))

hit_summary = hit_summary[hit_summary["hit_train"]>0.1]

print('Top 15 columns by hit_gap (mean across seeds):')
display(hit_summary.head(15).round(3))


In [ ]:
# Visual heatmap: hit_gap per (method, llm) and column
hm = hit_summary.copy()
hm['label'] = hm.apply(lambda r: r['method'] if r['llm'] == '-' else f"{r['method']}/{r['llm']}", axis=1)

# Keep only columns with at least some (positive) gap in one of the methods
cols_kept = (hm[hm['hit_gap'] > 0.005]['column'].unique().tolist()
             or hm['column'].unique().tolist())
hm = hm[hm['column'].isin(cols_kept)]

matrix = (hm.pivot_table(index='label', columns='column', values='hit_gap', aggfunc='mean')
            .fillna(0))
# Sort columns by maximum gap
matrix = matrix[matrix.max().sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(max(8, 0.6 * matrix.shape[1] + 2), max(2.5, 0.4 * matrix.shape[0] + 1.5)))
vmax = max(abs(matrix.values.min()), abs(matrix.values.max()))
im = ax.imshow(matrix.values, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xticks(np.arange(matrix.shape[1])); ax.set_xticklabels(matrix.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(np.arange(matrix.shape[0])); ax.set_yticklabels(matrix.index, fontsize=8)
ax.set_title(f'hit_gap = hit_train − hit_test at t={FOCAL_THRESHOLD} (mean across seeds)')
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)

# Annotate values
for r in range(matrix.shape[0]):
    for c in range(matrix.shape[1]):
        v = matrix.values[r, c]
        if abs(v) > 0.005:
            ax.text(c, r, f'{v:.2f}', ha='center', va='center',
                    color='white' if abs(v) > vmax/2 else 'black', fontsize=7)
fig.tight_layout(); plt.show()


## 8. Value Overlap
For a chosen column (by default the one with the largest `lift_gap` from section 7), measures `|syn ∩ real| / |syn|` against train and against test. **overlap_train ≫ overlap_test** → the synthetic values come specifically from the training set; **overlap_train ≈ overlap_test** → marginal coverage.

In [ ]:
# Column chosen automatically: largest lift_gap, preferring numeric ones
_num_top = lift_pivot[lift_pivot['dtype'] == 'numeric'].dropna(subset=['lift_train'])
OVERLAP_COLUMN = (_num_top.iloc[0]['column']
                  if not _num_top.empty
                  else lift_pivot.iloc[0]['column'])
print(f'Chosen column: {OVERLAP_COLUMN!r}')

overlap_df = se.run_value_overlap_evaluation(
    train_datasets, methods, seeds,
    test_datasets=test_datasets,
    columns=[OVERLAP_COLUMN],
)

overlap_pivot = (overlap_df
                 .pivot_table(index=['method', 'llm'], columns='reference',
                              values='overlap', aggfunc='mean')
                 .reset_index()
                 .rename(columns={'train': 'overlap_train', 'test': 'overlap_test'}))
overlap_pivot['overlap_gap'] = overlap_pivot['overlap_train'] - overlap_pivot['overlap_test']
overlap_pivot = overlap_pivot.sort_values('overlap_gap', ascending=False)
display(overlap_pivot.round(3))


In [ ]:
# Plot: overlap_train vs overlap_test (grouped bars)
agg = overlap_pivot.copy()
agg['label'] = agg.apply(lambda r: r['method'] if r['llm'] == '-' else f"{r['method']}/{r['llm']}", axis=1)

x = np.arange(len(agg))
w = 0.4
fig, ax = plt.subplots(figsize=(max(6, 0.6*len(agg)), 4))
ax.bar(x - w/2, agg['overlap_train'], width=w, label='overlap_train', color='C0')
ax.bar(x + w/2, agg['overlap_test'],  width=w, label='overlap_test',  color='C1')
ax.set_xticks(x); ax.set_xticklabels(agg['label'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel(f'|syn ∩ real| / |syn| in "{OVERLAP_COLUMN}"')
ax.set_ylim(0, 1.05)
ax.set_title(f'Value overlap in column {OVERLAP_COLUMN!r}')
ax.legend(); fig.tight_layout(); plt.show()


## 9. Save results to CSV
Consolidates all the scalar metrics into a single `(method, llm, seed)` table and exports to CSV. Per-column decomposition and per-threshold curve are exported to separate files.

In [ ]:
# Summary table: one row per (method, llm, seed) with all the scalar metrics
def _to_index(df):
    return df.set_index(['method', 'llm', 'seed'])

# Separate the discriminator i.i.d. baseline before joining it to the table
_disc_syn      = disc_df[disc_df['method'] != 'real_test']
_disc_baseline = (disc_df[disc_df['method'] == 'real_test']
                  .set_index('seed')['disc_auc']
                  .rename('disc_auc_baseline'))
_corr_syn      = corr_df[corr_df['method'] != 'real_test']
_corr_baseline = (corr_df[corr_df['method'] == 'real_test']
                  .set_index('seed')['corr_mae']
                  .rename('corr_mae_baseline'))

frames = [
    _to_index(_disc_syn)[['disc_auc']],
    _to_index(_corr_syn)[['corr_mae']],
    _to_index(equal_df)[['pct']].rename(columns={'pct': 'pct_equal_rows'}),
    _to_index(dcr_df)[['dcr_median', 'dcr_mean', 'dcr_5th', 'dcr_min']],
]

# Add curve (vs train) and premiums — only for rows that exist
curve_train = (curve_df[(curve_df['method'] != 'real_test') & (curve_df['reference'] == 'train')]
               .set_index(['method', 'llm', 'seed'])[pct_cols])
curve_test  = (curve_df[(curve_df['method'] != 'real_test') & (curve_df['reference'] == 'test')]
               .set_index(['method', 'llm', 'seed'])[pct_cols]
               .rename(columns={c: f'{c}_test' for c in pct_cols}))
premium_per_seed = syn.set_index(['method', 'llm', 'seed'])[premium_cols]
frames += [curve_train, curve_test, premium_per_seed]

summary = pd.concat(frames, axis=1).reset_index()
if not _disc_baseline.empty:
    summary['disc_auc_baseline'] = summary['seed'].map(_disc_baseline)
if not _corr_baseline.empty:
    summary['corr_mae_baseline'] = summary['seed'].map(_corr_baseline)

method_order = ['great', 'tabula', 'taptap'] + GANS
llm_order    = ['distilgpt2', 'taptap_distill', 'qwen3_03B_distil', '-']

method_cat = pd.Categorical(summary['method'], categories=method_order, ordered=True)
llm_cat    = pd.Categorical(summary['llm'],    categories=llm_order,    ordered=True)

summary = summary.assign(method=method_cat, llm=llm_cat).sort_values(['method', 'llm']).reset_index(drop=True)

summary_path = f'{RESULTS_DIR}/{dataset_name}_similarity_summary.csv'
summary.to_csv(summary_path, index=False)
print(f'Summary saved: {summary_path} (shape {summary.shape})')
display(summary.head().round(4))


In [ ]:
# Per-column decomposition (long format)
decomp_path = f'{RESULTS_DIR}/{dataset_name}_similarity_column_decomposition.csv'
decomp_df.to_csv(decomp_path, index=False)
print(f'Per-column decomposition saved: {decomp_path} (shape {decomp_df.shape})')

# Full curve (long format)
curve_path = f'{RESULTS_DIR}/{dataset_name}_similarity_threshold_curve.csv'
curve_df.to_csv(curve_path, index=False)
print(f'Threshold curve saved: {curve_path} (shape {curve_df.shape})')
# Robust per-(method, column) summary: hit_train, hit_test, hit_gap, hit_ratio
hit_summary_path = f'{RESULTS_DIR}/{dataset_name}_similarity_hit_rate_summary.csv'
hit_summary.head(15).to_csv(hit_summary_path, index=False)
print(f'Hit-rate summary saved: {hit_summary_path} (shape {hit_summary.shape})')

